# Финальный технический аудит

Интерактивная замена `final_audit.py`. Ноутбук сверяет сохранённые результаты без обучения, инференса или перезапуска bootstrap.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

ROOT = Path('reports') if Path('reports').exists() else Path('../reports')
FINAL = ROOT / 'final_evaluation'; AUDIT = ROOT / 'final_audit'
saved_metrics = {row['key']: row for row in json.loads((FINAL / 'metrics.json').read_text(encoding='utf-8'))}


In [ ]:
def recalculate(frame):
    y, p = frame.true_label.to_numpy(), frame.probability_ai.to_numpy(); pred = (p >= .5).astype(int)
    return {'accuracy': accuracy_score(y,pred), 'precision_label_1': precision_score(y,pred,zero_division=0), 'recall_label_1': recall_score(y,pred,zero_division=0), 'f1_label_1': f1_score(y,pred,zero_division=0), 'roc_auc': roc_auc_score(y,p), 'confusion_matrix': confusion_matrix(y,pred,labels=[0,1]).tolist()}

rows, ids, labels = [], None, None
for key, saved in saved_metrics.items():
    frame = pd.read_csv(FINAL / f'{key}_predictions.csv')
    current = recalculate(frame)
    assert all(np.isclose(current[m], saved[m]) for m in ['accuracy','precision_label_1','recall_label_1','f1_label_1','roc_auc'])
    assert current['confusion_matrix'] == saved['confusion_matrix']
    if ids is None: ids, labels = frame.id.to_numpy(), frame.true_label.to_numpy()
    else: assert np.array_equal(ids, frame.id) and np.array_equal(labels, frame.true_label)
    rows.append({'model': saved['model'], **{m: current[m] for m in ['accuracy','precision_label_1','recall_label_1','f1_label_1','roc_auc']}})
pd.DataFrame(rows)


In [ ]:
assignments = pd.read_csv(ROOT / 'baseline' / 'split_assignments.csv')
metadata = json.loads((ROOT / 'baseline' / 'split_metadata.json').read_text(encoding='utf-8'))
assert len(assignments) == 30000 and assignments.id.nunique() == 30000
assert set(assignments[assignments.subset == 'train'].id).isdisjoint(set(assignments[assignments.subset == 'internal_validation'].id))
assignments.groupby(['subset','label']).size(), metadata


In [ ]:
before = pd.read_csv(AUDIT / 'artifact_hashes_before.csv').rename(columns={'Path':'path','Hash':'sha256'})
mismatches = []
for _, row in before.iterrows():
    digest = hashlib.sha256(Path(row.path).read_bytes()).hexdigest().lower()
    if digest != row.sha256.lower(): mismatches.append(row.path)
print('Несовпадений SHA-256:', len(mismatches))
assert not mismatches


In [ ]:
pairs = pd.read_csv(ROOT / 'error_analysis' / 'bootstrap_pairwise_differences.csv')
assert len(pairs) == 4
pairs
# Интервалы ΔF1 включают ноль; интервалы ΔROC-AUC для указанных сравнений — нет.
